In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [4]:
from langchain_community.document_loaders import PyPDFLoader

c:\Users\athir\miniconda3\envs\qanda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
cd d:/Projects/Question-Answer-Creator-Application/

d:\Projects\Question-Answer-Creator-Application


In [6]:
file_path = "data/statistics.pdf"
loader = PyPDFLoader(file_path)
data = loader.load()
data

[Document(metadata={'producer': 'Skia/PDF m120 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Report - Importance and the use of correlation in Statistics', 'source': 'data/statistics.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='ImportanceandtheuseofcorrelationinStatistics\nIntroduction\nCorrelationisastatistical measurethat expressestheextent towhichtwovariablesarelinearlyrelated. It isacommontool for describingsimplerelationshipswithout makingastatement about causeandeffect.Correlationcoefficientsrangefrom-1to1, withavalueof 0indicatingnolinear relationshipbetweenthetwovariables, avalueof 1indicatingaperfect positivelinear relationship, andavalueof -1indicatingaperfectnegativelinear relationship.\nCorrelationisimportantinstatisticsbecauseitcanbeusedto\n1. Identifyrelationshipsbetweenvariables: Correlationcanbeusedtoidentifywhether thereisarelationshipbetweentwovariables, andif so, whether therelationshipispositiveornegative. Thisinfor

In [7]:
content = ""
for page in data:
    content += page.page_content
content

'ImportanceandtheuseofcorrelationinStatistics\nIntroduction\nCorrelationisastatistical measurethat expressestheextent towhichtwovariablesarelinearlyrelated. It isacommontool for describingsimplerelationshipswithout makingastatement about causeandeffect.Correlationcoefficientsrangefrom-1to1, withavalueof 0indicatingnolinear relationshipbetweenthetwovariables, avalueof 1indicatingaperfect positivelinear relationship, andavalueof -1indicatingaperfectnegativelinear relationship.\nCorrelationisimportantinstatisticsbecauseitcanbeusedto\n1. Identifyrelationshipsbetweenvariables: Correlationcanbeusedtoidentifywhether thereisarelationshipbetweentwovariables, andif so, whether therelationshipispositiveornegative. Thisinformationcanbeuseful for understandingtherelationshipsbetweendifferent factorsinacomplexsystem.\n2. Makepredictions: If thereisastrongcorrelationbetweentwovariables, it ispossibletousethevalueof onevariabletopredictthevalueof theother variable. Thiscanbeuseful for makingprediction

In [8]:
from langchain_text_splitters import TokenTextSplitter
from transformers import AutoTokenizer

# Load a specific Hugging Face tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [9]:
# Create the splitter using the loaded tokenizer's encode method
text_splitter = TokenTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=200,
    chunk_overlap=50
)

In [10]:
# Split your content
chunks = text_splitter.split_text(content)

In [11]:
print(len(content),len(chunks))

3664 6


In [12]:
from langchain_core.documents import Document

# convert string chunks to Document format
document_chunks = [Document(page_content=chunk) for chunk in chunks]

In [13]:
document = Document(page_content=content)

In [14]:
from langchain_groq import ChatGroq

llamaChatModel = ChatGroq(model="llama-3.3-70b-versatile",temperature = 0.3)
openaiChatModel =  ChatGroq(model="openai/gpt-oss-120b",temperature = 0.3)
qwenChatModel =  ChatGroq(model="qwen/qwen3-32b",temperature = 0.3)

For question generation, we directly pass content to model. For Aswer generation, we store the content in vector database and do retrieval.

In [30]:
question_prompt_template = """
You are an expert at creating questions based on given materials and documentation.
Your goal is to prepare a student for their exam by generating 10 questions.
You do this by asking questions about the text below:

------------
{text}
------------

Create questions that will prepare the students for their exams.
Make sure not to loose any important information.
Respond with only questions. No other information.

QUESTIONS:
"""

In [16]:
from langchain_core.prompts import PromptTemplate

question_prompt = PromptTemplate(template = question_prompt_template, input_variables = ["text"])

In [31]:
refine_question_template = """
You are an expert at creating questions based on materials and documentation.
Your goal is to help a student to prepare for their exam by generating 10 questions.
We have recieved some questions to a certain extent : 
{existing_answer}

We have the option to refine the existing questions or add the new ones deleting irrelevant questions (only if necessary) with some more context below.

------------
{text}
------------

Given the new context, refine the original questions in English.
If the context is not helpful, please provide the original question.
Respond with only 10 questions. No other information.

QUESTIONS:
"""

In [18]:
refine_question_prompt = PromptTemplate(
    input_variables = ["existing_answer","text"],
    template = refine_question_template
)

## Legacy chain

In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

question_chain = load_summarize_chain(
    llm = openaiChatModel,
    chain_type = "refine",
    verbose = True,
    question_prompt = question_prompt,
    refine_prompt = refine_question_prompt
)

questions = question_chain.run([document])
print(questions)

## LECL

In [32]:
from langchain_core.output_parsers import StrOutputParser

# Define the chains using LCEL
initial_chain = question_prompt | openaiChatModel | StrOutputParser()

refine_chain = refine_question_prompt | openaiChatModel | StrOutputParser()

print("Generating questions using LCEL pattern...")

# Start with the first chunk
current_questions = initial_chain.invoke({"text": document_chunks[0].page_content})

# Iteratively refine with the rest of the chunks
for i, doc in enumerate(document_chunks[1:], start=2):
    print(f"Refining with chunk {i}/{len(document_chunks)}...")
    current_questions = refine_chain.invoke({
        "existing_answer": current_questions,
        "text": doc.page_content
    })

print("\nFinal Questions Generated!")
print(current_questions)

Generating questions using LCEL pattern...
Refining with chunk 2/6...
Refining with chunk 3/6...
Refining with chunk 4/6...
Refining with chunk 5/6...
Refining with chunk 6/6...

Final Questions Generated!
1. What does a correlation coefficient of 0 tell us about the existence and direction of a linear relationship between two variables, such as a company’s advertising spend and its sales revenue?  

2. How do correlation coefficients of +1 and –1 differ in terms of direction (positive versus negative) and the strength of the linear relationship they represent, for example when comparing the returns of two stocks in a diversified portfolio?  

3. Why is correlation considered a useful descriptive tool for identifying relationships—e.g., between smoking prevalence and lung‑cancer incidence—without implying that one variable causes the other?  

4. In what ways can correlation be employed to detect and describe relationships among multiple business performance metrics (e.g., sales, custo

In [20]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [21]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(document_chunks,embedding_model)

In [22]:
questions = current_questions.split("\n\n")
questions

['1. What does a correlation coefficient measure, and how does it quantify both the strength and direction of the linear relationship between two variables?  ',
 '2. Why does a high correlation not imply a cause‑and‑effect relationship, and which logical fallacies (e.g., “post hoc ergo propter hoc”) arise when correlation is mistaken for causation?  ',
 '3. What is the possible range of values for a correlation coefficient, and what do the extreme values (‑1,\u202f0,\u202f+1) indicate about the relationship between the variables?  ',
 '4. What does a correlation coefficient of\u202f0 tell us about the linear association between two variables, and can a non‑linear relationship still exist despite this value?  ',
 '5. What does a correlation coefficient of\u202f+1 reveal about the nature of the relationship between two variables?  ',
 '6. What does a correlation coefficient of\u202f‑1 reveal about the nature of the relationship between two variables?  ',
 '7. Why is correlation considere

## Legacy chain

In [23]:
from langchain_classic.chains import RetrievalQA

answer_chain = RetrievalQA.from_chain_type(
    llm = openaiChatModel,
    chain_type = "stuff",
    retriever = vector_store.as_retriever()
)

## LCEL

In [24]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [25]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template(
    """Answer the question based only on the context below:

{context}

Question: {question}
"""
)

retriever = vector_store.as_retriever()

answer_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | openaiChatModel
    | StrOutputParser()
)

In [26]:
# Answer each question and save to a file

for question in questions:
    print(question)
    answer = answer_chain.invoke(question)
    print("Answer:",answer)
    print("------------------------------")
    with open("answers.txt","a") as file:
        file.write(question+"\n")
        file.write("Answer:"+answer+"\n")
        file.write("-------------------------------\n")

1. What does a correlation coefficient measure, and how does it quantify both the strength and direction of the linear relationship between two variables?  
Answer: A correlation coefficient is a single numeric value that tells how two variables move together in a **linear** way.  

* **What it measures:** It measures the degree to which changes in one variable are associated with proportional changes in another variable. In other words, it captures the extent of linear relationship between the two variables.  

* **How it quantifies strength:** The absolute size of the coefficient (its distance from 0) indicates how strong that linear relationship is. Values close to 1 or ‑1 (e.g., 0.8, ‑0.9) mean a strong linear association; values near 0 (e.g., 0.1, ‑0.2) mean a weak or negligible linear association.  

* **How it quantifies direction:** The sign of the coefficient tells the direction of the relationship.  
  * A **positive** coefficient (greater than 0) means that as one variable i

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km2c0ykgeaxt7519vka2yjfr` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6232, Requested 1832. Please try again in 480ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}